# INFOMDSS Team Project Examples

This Docker container may serve as a starting point for your course project.
It includes a set of simple instructions to:
<br>
 -> load a dataset (locally)
 <br>
 -> into a database
 <br>
 -> and be able to query data from the database
 <br>
 -> and make simple visualizations on the queried data

In [22]:
# Imports

from sqlalchemy import create_engine, text, inspect, Table
import pandas as pd
import numpy as np
import plotly.express as px
import matplotlib.pyplot as plt

## Load csv file

Load the file called world_population.csv into a pandas dataframe. Make sure you parse the columns correctly.

In [23]:
import os

# Get the current working directory
current_directory = os.getcwd()

# Get the parent directory (directory above the current directory)
parent_directory = os.path.dirname(current_directory)

# List all folders in the parent directory
folders_in_parent_directory = [folder for folder in os.listdir(parent_directory) if os.path.isdir(os.path.join(parent_directory, folder))]

# Print the list of folders
print("Folders in the parent directory:")
for folder in folders_in_parent_directory:
    print(folder)

Folders in the parent directory:
code
.vs
dashboard
notebook
.git
data


In [24]:
# Load the csv into a pandas dataframe (https://www.w3schools.com/python/pandas/pandas_dataframes.asp)
ams_aqi = pd.read_csv("../data/amsterdam-air-quality.csv", index_col=0, parse_dates=True)
bxl_aqi = pd.read_csv("../data/brussels-air-quality.csv", index_col=0, parse_dates=True)
hel_aqi = pd.read_csv("../data/helsinki-air-quality.csv", index_col=0, parse_dates=True)
ldn_aqi = pd.read_csv("../data/london-air-quality.csv", index_col=0, parse_dates=True)
par_aqi = pd.read_csv("../data/paris-air-quality.csv", index_col=0, parse_dates=True)

# Data Cleaning

* Issue(solved) : original dataset comes with columns like this Index(['date', ' pm25', ' pm10', ' o3', ' no2', ' so2', ' co'], dtype='object'), which contains column names with space

In [25]:
def rename_column(df):
    """
    to modify column name because in each column there is a space
    """
    new_column_names = {' pm25': 'pm25',
                       ' pm10': 'pm10',
                       ' o3': 'o3',
                       ' no2': 'no2',
                       ' so2': 'so2',
                       ' co': 'co'
    }
    
    df.rename(columns=new_column_names, inplace=True)

def reformat_record(df): 
    """
    to change the format of date to the standardized expression, and change the format of records from string to float.
    return a dataset with clean format.
    """
    df['date'] = pd.to_datetime(df['date']) # correct the format of date
    df.set_index('date', inplace=True) # set the date as an index of ams_aqi 
    df.sort_index(inplace=True) # sort the dataset by the date index
    df = df.apply(pd.to_numeric, errors='coerce') # change the data type of rows from string to float
    return df

def data_imputation(df):
    """
    to fill up the missing value.
    # TODO : waiting for Nils to write a better method
    """
    columns_to_fill = ['pm25', 'pm10', 'o3', 'no2'] # specify in which column, imputation is required
    
    for column in columns_to_fill:
        df[column].fillna(method='ffill', inplace=True) # fill the missing value with the previous value
    
    for column in columns_to_fill:
        df[column].fillna(df[column].mean(), inplace=True) # fill the rest of missing value with mean value

    

* Issue : Many missing values in SO2 and CO in depedning on each country

* Decision : Only use pm2.5, pm10, O3, NO2

## Store data into database
Save the contents in the world_population file to the a table called population in the database. 

In [8]:
# Create a SQLAlchemy engine to connect to the PostgreSQL database
engine = create_engine("postgresql://student:infomdss@db_dashboard:5432/dashboard")

# Establish a connection to the database using the engine
# The 'with' statement ensures that the connection is properly closed when done
with engine.connect() as conn:
    # Execute an SQL command to drop the 'population' table if it exists
    # The text() function allows you to execute raw SQL statements
    result = conn.execute(text("DROP TABLE IF EXISTS AQI_AMS CASCADE;"))
    result = conn.execute(text("DROP TABLE IF EXISTS AQI_BXL CASCADE;"))
    result = conn.execute(text("DROP TABLE IF EXISTS AQI_HEL CASCADE;"))
    result = conn.execute(text("DROP TABLE IF EXISTS AQI_LDN CASCADE;"))
    result = conn.execute(text("DROP TABLE IF EXISTS AQI_PAR CASCADE;"))
# Assuming you have a DataFrame named 'world_population_df', the following line
# writes the data from the DataFrame to a new 'population' table in the database
# If the 'population' table already exists, it will be replaced with the new data
# This prints the number of rows entered in the database table
ams_aqi.to_sql("AQI_AMS", engine, if_exists="replace", index=True)
bxl_aqi.to_sql("AQI_BXL", engine, if_exists="replace", index=True)
hel_aqi.to_sql("AQI_HEL", engine, if_exists="replace", index=True)
ldn_aqi.to_sql("AQI_LDN", engine, if_exists="replace", index=True)
par_aqi.to_sql("AQI_PAR", engine, if_exists="replace", index=True)

428

## Fetch data from database
Read the table **population** from the database in a dataframe. Make sure the index column is the index of the dataframe.

In [9]:
# Read data from the SQL table named 'population' using pandas
# 'pd.read_sql_table' is a pandas function that reads data from an SQL table
# 'db_conn' is the database connection object previously established
ams_aqi = pd.read_sql_table('AQI_AMS', engine, index_col='date')
bxl_aqi = pd.read_sql_table('AQI_BXL', engine, index_col='index')
hel_aqi = pd.read_sql_table('AQI_HEL', engine, index_col='index')
ldn_aqi = pd.read_sql_table('AQI_LDN', engine, index_col='index')
par_aqi = pd.read_sql_table('AQI_PAR', engine, index_col='index')
# This line prints the entire DataFrame to the output
print(ams_aqi)
print(bxl_aqi)
# Note that we transformed the data from a .csv file to a pandas dataframe
# Then loaded the dataframe into the database table
# And now we have pulled the data from the database and put it in a dataframe again
# This is an example of how you might store and fetch data to and from your database for your dashboard

            index  pm25  pm10   o3  no2  so2  co
date                                            
2023/9/1        0    43    24   33   14        3
2023/9/2        1    61    19   34   18        3
2023/9/3        2    41    22   39   19        4
2023/9/4        3    49    25   44   23        4
2023/9/5        4    47    37   62   29        4
...           ...   ...   ...  ...  ...  ...  ..
2014/3/30    3342          42   23   38    1    
2014/3/31    3343          56   27   47    6    
2014/1/29    3344                1         1    
2013/12/31   3345                    30         
2014/1/12    3346                    41         

[3347 rows x 7 columns]
             date  pm25  pm10   o3  no2  so2  co
index                                           
0        2023/9/1    25    13   26    7         
1        2023/9/2    36    21   38    6    1    
2        2023/9/3    66    26   42   12    1    
3        2023/9/4    53    27   38   17         
4        2023/9/5    44    30   49   19     

# Data Transformation

In [27]:
def data_transformation(df):
    """
    calculate monthly average of pm25, pm10, o3 and no2
    return monthly average dataframes
    """
    columns_to_average = ['pm25', 'pm10', 'o3', 'no2'] # specify the columns needed to be aggregated
    monthly_averages_df = pd.DataFrame() # init a empty dict for further appending 
    for column in columns_to_average:
        monthly_averages = df[column].resample('M').mean().round(2) # calculate the average value in each month
        monthly_averages_df[column] = monthly_averages # append a new column in the dictioary
        monthly_averages_df[column].fillna(monthly_averages_df[column].mean().round(2), inplace=True)
    return monthly_averages_df

In [28]:
monthly_ams_aqi = data_transformation(ams_aqi)
monthly_bxl_aqi = data_transformation(bxl_aqi)
monthly_hel_aqi = data_transformation(hel_aqi)
monthly_ldn_aqi = data_transformation(ldn_aqi)
monthly_par_aqi = data_transformation(par_aqi)

monthly_ams_aqi.to_csv("monthly_ams_aqi.csv", index=True)
monthly_bxl_aqi.to_csv("monthly_bxl_aqi.csv", index=True)
monthly_hel_aqi.to_csv("monthly_hel_aqi.csv", index=True)
monthly_ldn_aqi.to_csv("monthly_ldn_aqi.csv", index=True)
monthly_par_aqi.to_csv("monthly_par_aqi.csv", index=True)


In [29]:
monthly_ams_aqi

,pm25,pm10,o3,no2
date,,,,
2013-12-31,56.49,24.85,26.37,30.00
2014-01-31,56.49,25.20,10.00,32.58
2014-02-28,56.49,21.43,18.07,28.21
2014-03-31,56.49,36.42,20.97,32.39
2014-04-30,56.49,38.50,23.40,30.60
...,...,...,...,...
2023-05-31,48.94,23.39,33.65,14.19
2023-06-30,42.27,26.03,42.53,14.23
2023-07-31,34.52,17.90,30.97,10.39


## Plot data
We can meaningfully plot the data from the database in many different ways to give users of the dashboard a visual impression of the data. 
<br>
For the example data on the world population, we plot a line chart as an example. 

In [30]:
fig = px.line(monthly_ams_aqi, x=monthly_ams_aqi.index, y='pm25', title='Monthly Average PM2.5 Levels Over Time')
fig.show()